## Estimate number of tokens in test datasets for basic cost estimates on llm stuff

In [1]:
from pathlib import Path
import tiktoken

candidate_dirs = [
    "data/dataset/1_error/test",
    "data/dataset/2_error/test",
    "data/dataset/5_error/test",
]



def get_tokens_for_documents_in_dir(dir):
    files = []
    p = Path(dir)
    if p.exists():
        files = sorted(p.rglob("*.txt"))
    if not files:
        return 0

    encoding = tiktoken.get_encoding("cl100k_base")
    total_tokens = 0
    for fp in files:
        try:
            text = fp.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            text = fp.read_text(errors="ignore")
        total_tokens += len(encoding.encode(text))

    return total_tokens

total_tokens = sum(get_tokens_for_documents_in_dir(dirname) for dirname in candidate_dirs)

print(f"Total tokens (cl100k_base): {total_tokens}")


Total tokens (cl100k_base): 5183886


In [2]:
# TODO: stats here

In [5]:
from pathlib import Path
from collections import Counter
import re
import numpy as np
import pandas as pd

candidate_dirs = [
    "data/dataset/1_error/test",
    "data/dataset/2_error/test",
    "data/dataset/5_error/test",
    "data/dataset/1_error/train",
    "data/dataset/2_error/train",
    "data/dataset/5_error/train",
]

_token_pattern = re.compile(r"\b\w+\b")

def tokenize(text: str):
    return _token_pattern.findall(text.lower())


def parse_labels(header_line: str):
    # Extract integers from header line (e.g., "continuity [1, 3]")
    return [int(x) for x in re.findall(r"\d+", header_line)]


def compute_stats_for_dir(dir_path: str):
    files = sorted(Path(dir_path).rglob("*.txt"))
    if not files:
        return None

    story_lengths = []
    sentence_lengths = []
    token_counter = Counter()
    bigram_counter = Counter()
    label_pos_counter = Counter()

    stories = 0
    sentences_total = 0
    tokens_total = 0
    bigrams_total = 0
    labels_total = 0

    for fp in files:
        try:
            text = fp.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            text = fp.read_text(errors="ignore")
        lines = [ln.rstrip("\n") for ln in text.splitlines()]
        if not lines:
            continue
        header = lines[0]
        sentences = [ln for ln in lines[1:] if ln.strip()]

        stories += 1
        story_len = len(sentences)
        story_lengths.append(story_len)
        sentences_total += story_len

        labels = parse_labels(header)
        if labels:
            labels_total += len(labels)
            for pos in labels:
                label_pos_counter[pos] += 1

        for s in sentences:
            toks = tokenize(s)
            sentence_lengths.append(len(toks))
            tokens_total += len(toks)
            token_counter.update(toks)
            if len(toks) > 1:
                bigrams = list(zip(toks, toks[1:]))
                bigram_counter.update(bigrams)
                bigrams_total += len(bigrams)

    if stories == 0 or sentences_total == 0 or tokens_total == 0:
        return None

    def pctl(arr, p):
        return float(np.percentile(arr, p)) if arr else float("nan")

    stats = {
        "stories": stories,
        "sentences_total": sentences_total,
        "avg_sents_per_story": float(np.mean(story_lengths)),
        "p10_sents_per_story": pctl(story_lengths, 10),
        "median_sents_per_story": float(np.median(story_lengths)),
        "p90_sents_per_story": pctl(story_lengths, 90),
        "avg_tokens_per_sentence": float(np.mean(sentence_lengths)) if sentence_lengths else float("nan"),
        "median_tokens_per_sentence": float(np.median(sentence_lengths)) if sentence_lengths else float("nan"),
        "vocab_size": len(token_counter),
        "tokens_total": tokens_total,
        "type_token_ratio": (len(token_counter) / tokens_total) if tokens_total else float("nan"),
        "distinct_1": (len(token_counter) / tokens_total) if tokens_total else float("nan"),
        "distinct_2": (len(bigram_counter) / bigrams_total) if bigrams_total else float("nan"),
        "labels_total": labels_total,
        "label_rate_per_sentence": (labels_total / sentences_total) if sentences_total else float("nan"),
        "avg_errors_per_story": (labels_total / stories) if stories else float("nan"),
        "top_error_positions": ", ".join(
            f"{pos}:{cnt}" for pos, cnt in sorted(label_pos_counter.items(), key=lambda x: (-x[1], x[0]))[:5]
        ),
    }

    # Also return raw counters for an aggregate pass
    raw = {
        "story_lengths": story_lengths,
        "sentence_lengths": sentence_lengths,
        "token_counter": token_counter,
        "bigram_counter": bigram_counter,
        "label_pos_counter": label_pos_counter,
        "stories": stories,
        "sentences_total": sentences_total,
        "tokens_total": tokens_total,
        "bigrams_total": bigrams_total,
        "labels_total": labels_total,
    }
    return stats, raw


per_dir_rows = []
raws = []
for d in candidate_dirs:
    res = compute_stats_for_dir(d)
    if res is None:
        continue
    stats, raw = res
    stats_row = {"dataset": d} | stats
    per_dir_rows.append(stats_row)
    raws.append(raw)

# Aggregate across all selected dirs
if raws:
    agg = {
        "story_lengths": sum((r["story_lengths"] for r in raws), []),
        "sentence_lengths": sum((r["sentence_lengths"] for r in raws), []),
        "token_counter": sum((r["token_counter"] for r in raws), Counter()),
        "bigram_counter": sum((r["bigram_counter"] for r in raws), Counter()),
        "label_pos_counter": sum((r["label_pos_counter"] for r in raws), Counter()),
        "stories": sum(r["stories"] for r in raws),
        "sentences_total": sum(r["sentences_total"] for r in raws),
        "tokens_total": sum(r["tokens_total"] for r in raws),
        "bigrams_total": sum(r["bigrams_total"] for r in raws),
        "labels_total": sum(r["labels_total"] for r in raws),
    }

    def pctl(arr, p):
        return float(np.percentile(arr, p)) if arr else float("nan")

    all_stats = {
        "dataset": "ALL",
        "stories": agg["stories"],
        "sentences_total": agg["sentences_total"],
        "avg_sents_per_story": float(np.mean(agg["story_lengths"])) if agg["story_lengths"] else float("nan"),
        "p10_sents_per_story": pctl(agg["story_lengths"], 10),
        "median_sents_per_story": float(np.median(agg["story_lengths"])) if agg["story_lengths"] else float("nan"),
        "p90_sents_per_story": pctl(agg["story_lengths"], 90),
        "avg_tokens_per_sentence": float(np.mean(agg["sentence_lengths"])) if agg["sentence_lengths"] else float("nan"),
        "median_tokens_per_sentence": float(np.median(agg["sentence_lengths"])) if agg["sentence_lengths"] else float("nan"),
        "vocab_size": len(agg["token_counter"]),
        "tokens_total": agg["tokens_total"],
        "type_token_ratio": (len(agg["token_counter"]) / agg["tokens_total"]) if agg["tokens_total"] else float("nan"),
        "distinct_1": (len(agg["token_counter"]) / agg["tokens_total"]) if agg["tokens_total"] else float("nan"),
        "distinct_2": (len(agg["bigram_counter"]) / agg["bigrams_total"]) if agg["bigrams_total"] else float("nan"),
        "labels_total": agg["labels_total"],
        "label_rate_per_sentence": (agg["labels_total"] / agg["sentences_total"]) if agg["sentences_total"] else float("nan"),
        "avg_errors_per_story": (agg["labels_total"] / agg["stories"]) if agg["stories"] else float("nan"),
        "top_error_positions": ", ".join(
            f"{pos}:{cnt}" for pos, cnt in sorted(agg["label_pos_counter"].items(), key=lambda x: (-x[1], x[0]))[:5]
        ),
    }
    per_dir_rows.append(all_stats)

if per_dir_rows:
    df = pd.DataFrame(per_dir_rows)
    cols = [
        "dataset",
        "stories",
        "sentences_total",
        "avg_sents_per_story",
        "median_sents_per_story",
        "p10_sents_per_story",
        "p90_sents_per_story",
        "avg_tokens_per_sentence",
        "median_tokens_per_sentence",
        "vocab_size",
        "tokens_total",
        "type_token_ratio",
        "distinct_1",
        "distinct_2",
        "labels_total",
        "label_rate_per_sentence",
        "avg_errors_per_story",
        "top_error_positions",
    ]
    # Keep only available columns (in case of missing data)
    cols = [c for c in cols if c in df.columns]
    display(df[cols].reset_index(drop=True))
else:
    print("No data files found in candidate_dirs:")
    for d in candidate_dirs:
        print(" -", d)



,dataset,stories,sentences_total,avg_sents_per_story,median_sents_per_story,p10_sents_per_story,p90_sents_per_story,avg_tokens_per_sentence,median_tokens_per_sentence,vocab_size,tokens_total,type_token_ratio,distinct_1,distinct_2,labels_total,label_rate_per_sentence,avg_errors_per_story,top_error_positions
0,data/dataset/1_error/test,2000,116198,58.099000,57.0,31.0,88.0,11.698342,10.0,25632,1359324,0.018856,0.018856,0.173724,2000,0.017212,1.0000,"6:54, 14:49, 2:47, 17:47, 9:46"
1,data/dataset/2_error/test,2000,118169,59.084500,56.0,31.0,89.0,11.638899,10.0,25793,1375357,0.018754,0.018754,0.174639,4000,0.033850,2.0000,"17:98, 1:94, 14:92, 20:91, 2:88"
2,data/dataset/5_error/test,2000,119292,59.646000,57.0,31.0,92.0,11.495322,9.0,22260,1371300,0.016233,0.016233,0.138630,9981,0.083669,4.9905,"2:228, 1:218, 0:211, 6:205, 8:203"
3,data/dataset/1_error/train,8000,477870,59.733750,57.0,32.0,91.0,11.489874,9.0,26444,5490666,0.004816,0.004816,0.047222,8000,0.016741,1.0000,"10:181, 12:173, 5:167, 1:162, 7:160"
4,data/dataset/2_error/train,8000,478842,59.855250,58.0,32.0,91.0,11.484404,9.0,26489,5499215,0.004817,0.004817,0.047789,15992,0.033397,1.9990,"20:338, 2:334, 9:324, 5:322, 3:321"
5,data/dataset/5_error/train,8000,479967,59.995875,58.0,32.0,91.0,11.497320,9.0,26561,5518334,0.004813,0.004813,0.048873,39968,0.083272,4.9960,"17:834, 10:829, 7:824, 2:805, 8:796"
6,ALL,30000,1790338,59.677933,57.0,32.0,91.0,11.514136,9.0,36498,20614196,0.001771,0.001771,0.022143,79941,0.044651,2.6647,"10:1641, 2:1639, 1:1616, 17:1614, 5:1590"
